<a href="https://colab.research.google.com/github/CheRongtian/Financial_Formula/blob/main/B_S_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Mar  9 17:11:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!curl https://developer.download.nvidia.com/hpc-sdk/ubuntu/DEB-GPG-KEY-NVIDIA-HPC-SDK | gpg --dearmor -o /usr/share/keyrings/nvidia-hpc-sdk-archive-keyring.gpg
!echo "deb [signed-by=/usr/share/keyrings/nvidia-hpc-sdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /" | tee /etc/apt/sources.list.d/nvhpc.list
!apt-get update -y
!apt-get install -y nvhpc-24-1

!pip install pybind11 yfinance pandas numpy

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1626  100  1626    0     0   2583      0 --:--:-- --:--:-- --:--:--  2585
deb [signed-by=/usr/share/keyrings/nvidia-hpc-sdk-archive-keyring.gpg] https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64 /
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:4 https://developer.download.nvidia.com/hpc-sdk/ubuntu/amd64  InRelease [2,126 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,426 kB]
Hit:8 http://archive.ubuntu.com/ubuntu jammy

In [9]:
!apt-get install -y python3-dev

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore libpython3.10
  libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib python3.10
  python3.10-dev python3.10-minimal
Suggested packages:
  apache2 | lighttpd | httpd python3.10-venv python3.10-doc binfmt-support
The following NEW packages will be installed:
  javascript-common libjs-sphinxdoc libjs-underscore python3-dev
  python3.10-dev
The following packages will be upgraded:
  libpython3.10 libpython3.10-dev libpython3.10-minimal libpython3.10-stdlib
  python3.10 python3.10-minimal
6 upgraded, 5 newly installed, 0 to remove and 103 not upgraded.
Need to get 13.0 MB of archives.
After this operation, 1,252 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libpython3.10-dev amd64 3.10.12-1~22.04.15 [4,764 kB]
Get:2 http://a

In [10]:
%%writefile montecarlo.cpp
#include <pybind11/pybind11.h>
#include <pybind11/stl.h>
#include <random>
#include <cmath>
#include <numeric>
#include <execution>
#include <vector>
#include <atomic>

namespace py = pybind11;

double SimpleMonteCarlo1(double Expiry, double Strike, double Spot, double Vol, double r, unsigned long NumberOfPaths) {
    double variance = Vol * Vol * Expiry;
    double rootVariance = sqrt(variance);
    double itoCorrection = -0.5 * variance;
    double moveSpot = Spot * exp(r * Expiry + itoCorrection);

    std::vector<double> random_gaussians(NumberOfPaths);
    std::mt19937 gen(std::random_device{}());
    std::normal_distribution<double> dist(0.0, 1.0);

    for (unsigned long i = 0; i < NumberOfPaths; ++i) {
        random_gaussians[i] = dist(gen);
    }

    std::vector<unsigned long> paths(NumberOfPaths);
    std::iota(paths.begin(), paths.end(), 0UL);

    const double* gaussians_ptr = random_gaussians.data();
    double runningSum = std::transform_reduce(
        std::execution::par_unseq,
        paths.begin(), paths.end(),
        0.0,
        std::plus<double>{},
        [=](unsigned long i) {
            double thisGaussian = gaussians_ptr[i];
            double thisSpot = moveSpot * exp(rootVariance * thisGaussian);
            double thisPayoff = thisSpot - Strike;
            return thisPayoff > 0.0 ? thisPayoff : 0.0;
        }
    );

    double mean = runningSum / NumberOfPaths;
    mean *= exp(-r * Expiry);
    return mean;
}

PYBIND11_MODULE(montecarlo, m) {
    m.doc() = "GPU Accelerated Monte Carlo Option Pricing Plugin";
    m.def("SimpleMonteCarlo1", &SimpleMonteCarlo1, "Price an option using Monte Carlo simulation");
}

Overwriting montecarlo.cpp


In [14]:
%%writefile main.py
import yfinance as yf
import datetime
import montecarlo

tickers = ["AAPL", "NVDA", "TSLA", "MSFT"]
risk_free_rate = 0.04
num_paths = 1000000

print(f"Initialize GPU parallel Monte Carlo...\npaths: {num_paths}\n" + "="*50)

for ticker_symbol in tickers:
    try:
        print(f"Extracting [{ticker_symbol}] data...")
        ticker = yf.Ticker(ticker_symbol)
        spot_price = ticker.info.get('currentPrice')

        if not spot_price:
            print(f"Cannot extract {ticker_symbol} price, skip\n" + "-"*40)
            continue

        options_dates = ticker.options
        if not options_dates:
            print(f"{ticker_symbol} do not have the data, skip\n" + "-"*40)
            continue

        expiry_date_str = options_dates[0]
        opt_chain = ticker.option_chain(expiry_date_str)
        calls = opt_chain.calls

        atm_call = calls.iloc[(calls['strike'] - spot_price).abs().argsort()[:1]].iloc[0]
        strike_price = atm_call['strike']
        implied_vol = atm_call['impliedVolatility']

        expiry_date = datetime.datetime.strptime(expiry_date_str, "%Y-%m-%d").date()
        today = datetime.date.today()
        #time_to_expiry = max((expiry_date - today).days / 365.0, 0.001)
        days_to_expiry = (expiry_date - today).days
        if days_to_expiry <= 0:
            days_to_expiry = 1

        time_to_expiry = days_to_expiry / 252.0

        calculated_price = montecarlo.SimpleMonteCarlo1(
            time_to_expiry, strike_price, spot_price, implied_vol, risk_free_rate, num_paths
        )

        print(f"[{ticker_symbol}] current price: ${spot_price} | exercise price: ${strike_price} | expire: {time_to_expiry:.4f}yr | IV: {implied_vol:.4f}")
        print(f"  => C++ GPU theorical price: ${calculated_price:.4f}")
        print(f"  => actual price: ${atm_call['lastPrice']:.4f}")
        print("-" * 40)

    except Exception as e:
        print(f"dealing with {ticker_symbol} happens errors: {e}\n" + "-"*40)

Overwriting main.py


In [16]:
!rm -f montecarlo*so montecarlo

!/opt/nvidia/hpc_sdk/Linux_x86_64/24.1/compilers/bin/nvc++ -O3 -shared -std=c++17 -stdpar=gpu -fPIC \
--diag_suppress=inline_gnu_noinline_conflict \
$(python3 -m pybind11 --includes) \
montecarlo.cpp \
-o montecarlo$(python3 -c "import sysconfig; print(sysconfig.get_config_var('EXT_SUFFIX'))")

!echo "=== check compile result ==="
!ls -l montecarlo*.so

!echo "=== start running monte carlo ==="
!python3 main.py

=== check compile result ===
-rwxr-xr-x 1 root root 1703968 Mar  9 17:42 montecarlo.cpython-312-x86_64-linux-gnu.so
=== start running monte carlo ===
Initialize GPU parallel Monte Carlo...
paths: 1000000
Extracting [AAPL] data...
[AAPL] current price: $256.3473 | exercise price: $257.5 | expire: 0.0040yr | IV: 0.2256
  => C++ GPU theorical price: $0.9646
  => actual price: $0.6900
----------------------------------------
Extracting [NVDA] data...
[NVDA] current price: $179.17 | exercise price: $180.0 | expire: 0.0040yr | IV: 0.2666
  => C++ GPU theorical price: $0.8429
  => actual price: $0.6100
----------------------------------------
Extracting [TSLA] data...
[TSLA] current price: $389.155 | exercise price: $390.0 | expire: 0.0040yr | IV: 0.2327
  => C++ GPU theorical price: $1.9072
  => actual price: $1.4800
----------------------------------------
Extracting [MSFT] data...
[MSFT] current price: $404.775 | exercise price: $405.0 | expire: 0.0040yr | IV: 0.3171
  => C++ GPU theorical